# 模型统一配置说明

## 配置一览

模型与密钥统一放在仓库根目录 `.env`，Notebook 用 `load_dotenv(find_dotenv())` 读取，代码里不写死地址与密钥。

| 能力 | 服务 | 环境变量 | 模型 |
| --- | --- | --- | --- |
| 大语言模型 | vLLM（OpenAI 兼容 `/v1`） | `vLLM_QWEN_BASE_URL` / `vLLM_QWEN_API_KEY` / `vLLM_QWEN_MODEL_ID` | `my_qwen3.8_flash_next` |
| Embedding | Xinference（`/v1/embeddings`） | `XINFERENCE_OPENAI_BASE_URL` / `XINFERENCE_API_KEY` / `XINFERENCE_EMBEDDING_MODEL_ID` | `my_qwen3_embedding_0.6b`（1024 维） |
| Reranker | Xinference（`/v1/rerank`） | `XINFERENCE_OPENAI_BASE_URL` / `XINFERENCE_API_KEY` / `XINFERENCE_RERANKER_MODEL_ID` | `my_qwen3_reranker_0.6b` |
| 向量数据库 | Milvus standalone（本机 Docker） | `MILVUS_URI` | collection `rag_lab_demo`（演示用） |

Embedding 与 Reranker 由同一个 Xinference 实例提供：`XINFERENCE_BASE_URL` 是原生 REST 根地址（给 REST 客户端用），`XINFERENCE_OPENAI_BASE_URL` 是它的 `/v1` 兼容地址（给 OpenAI 风格客户端用），两者共用 `XINFERENCE_API_KEY`。

Embedding 和 Reranker 是远端 HTTP 服务，Milvus 跑在本机 Docker，三类服务互不影响：换模型或换向量库地址都只改 `.env`。

本 Notebook 用 `env_rag_uv` 内核（`~/env_rag_uv`）实跑通过，每段代码后面的 **预期输出** 就是实跑结果；大语言模型的回答是生成式的，字句会略有出入，意思一致即可。


代理清除，如果遇到代理冲突报错

In [1]:
import warnings
warnings.filterwarnings("ignore")


import os
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy"
]:
    os.environ.pop(k, None)


## 大语言模型：vLLM（OpenAI 兼容）

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

_ = load_dotenv(find_dotenv())

client = OpenAI(
    base_url=os.getenv("vLLM_QWEN_BASE_URL"),
    api_key=os.getenv("vLLM_QWEN_API_KEY"),
)

response = client.chat.completions.create(
    model=os.getenv("vLLM_QWEN_MODEL_ID"),
    messages=[
        {"role": "system", "content": "你是一个 RAG 助教"},
        {"role": "user", "content": "用一句话说明什么是 RAG"}
    ],
    temperature=0.3,
    max_tokens=256,
    stream=False,
)

print(response.choices[0].message.content)




RAG（检索增强生成）是一种让大模型在回答问题前先检索外部知识库，再基于检索到的信息生成答案的技术。


**预期输出：**

```text
RAG（检索增强生成）是一种让大模型在回答问题前先从外部知识库检索相关资料，再基于这些资料生成答案的技术。
```

回答由模型生成，字句会略有不同，意思一致即为通过。


In [3]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

_ = load_dotenv(find_dotenv())

my_llm = init_chat_model(
    model=os.getenv("vLLM_QWEN_MODEL_ID"),
    model_provider="openai",
    base_url=os.getenv("vLLM_QWEN_BASE_URL"),
    api_key=os.getenv("vLLM_QWEN_API_KEY"),
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=4096,
)

print(my_llm.invoke("用一句话说明什么是 RAG").content)




RAG（检索增强生成）是一种让大模型在回答问题前先从外部知识库检索相关信息，再基于这些资料生成答案的技术。


**预期输出：**

```text
RAG（检索增强生成）是一种让大模型在回答问题前先从外部知识库检索相关资料，再基于这些资料生成更准确、可溯源答案的技术。
```

同上，回答内容与上一段一致即可；`init_chat_model` 让后续课件可以只换模型不改代码。


## Embedding 模型：Xinference（OpenAI 兼容 `/v1/embeddings`）

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import OpenAIEmbeddings

_ = load_dotenv(find_dotenv())

my_emb = OpenAIEmbeddings(
    base_url=os.getenv("XINFERENCE_OPENAI_BASE_URL"),
    api_key=os.getenv("XINFERENCE_API_KEY"),
    model=os.getenv("XINFERENCE_EMBEDDING_MODEL_ID"),
)

vec = my_emb.embed_query("人工智能是什么")

print(len(vec))
print(vec[:5])


1024
[-0.048095703125, -0.032470703125, -0.0081787109375, -0.0810546875, -0.0556640625]


**预期输出：**

```text
1024
[-0.048095703125, -0.032470703125, -0.0081787109375, -0.0810546875, -0.0556640625]
```

维度必须与 Milvus collection 的向量维度一致（这里 1024），换 Embedding 模型要重建 collection。


## Reranker 模型：Xinference（OpenAI 兼容 `/v1/rerank`）

In [5]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

query = "什么是人工智能"
documents = [
    "人工智能是研究如何让机器模拟人类智能",
    "今天的天气很好",
]

# Xinference 暴露 OpenAI 兼容的 /v1 接口，rerank 走 /v1/rerank
resp = httpx.post(
    os.environ["XINFERENCE_OPENAI_BASE_URL"].rstrip("/") + "/rerank",
    headers={"Authorization": "Bearer " + os.environ["XINFERENCE_API_KEY"]},
    json={
        "model": os.environ["XINFERENCE_RERANKER_MODEL_ID"],
        "query": query,
        "documents": documents,
        "top_n": len(documents),
    },
    timeout=60,
)
resp.raise_for_status()

for item in resp.json()["results"]:
    print(item["index"], round(item["relevance_score"], 6), documents[item["index"]])


0 1.0 人工智能是研究如何让机器模拟人类智能
1 9e-05 今天的天气很好


**预期输出：**

```text
0 1.0 人工智能是研究如何让机器模拟人类智能
1 9e-05 今天的天气很好
```

相关文档的 `relevance_score` 明显高于无关文档（1.0 对 9e-05），说明 Reranker 生效。


## 备选：Xinference 原生 REST 客户端

In [6]:
# 备选写法：Xinference 原生 REST 客户端（依赖 xinference-client，本项目锁 1.13.0）
import os
from dotenv import load_dotenv, find_dotenv
from xinference_client import RESTfulClient

_ = load_dotenv(find_dotenv())

xf = RESTfulClient(
    os.environ["XINFERENCE_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
)

for model_id, info in xf.list_models().items():
    print(model_id, info["model_type"], info.get("dimensions"))

reranker = xf.get_model(os.environ["XINFERENCE_RERANKER_MODEL_ID"])
result = reranker.rerank(documents=documents, query=query)
print([(item["index"], round(item["relevance_score"], 6)) for item in result["results"]])


my_qwen3_reranker_0.6b rerank None
my_qwen3_embedding_0.6b embedding 1024


[(0, 1.0), (1, 9e-05)]


**预期输出：**

```text
my_qwen3_reranker_0.6b rerank None
my_qwen3_embedding_0.6b embedding 1024
[(0, 1.0), (1, 9e-05)]
```

原生客户端与 `/v1` 兼容接口看到的是同一批模型；`xinference-client` 1.13 起模块名是 `xinference_client`，类名 `RESTfulClient`。


## 向量数据库：Milvus（本机 Docker）

向量库地址同样写在 `.env`：`MILVUS_URI`，本机 Docker 单机版就是 `http://localhost:19530`；Attu 管理面板在 `http://localhost:16630`，可以在图形界面里看 collection、Segment 和标量过滤。

客户端用 `pymilvus` 3.x，和服务端 3.x 保持同大版本。单机版默认不开鉴权；如果开启了鉴权，在 `.env` 补一行 `MILVUS_TOKEN = "user:password"`，再给 `MilvusClient(uri=..., token=...)`。

In [7]:
import os
from dotenv import load_dotenv, find_dotenv
from pymilvus import MilvusClient

_ = load_dotenv(find_dotenv())

milvus_uri = os.environ["MILVUS_URI"]

client = MilvusClient(uri=milvus_uri)

print(milvus_uri)
print(client.get_server_version())

http://localhost:19530
3.0.3


**预期输出：**

```text
http://localhost:19530
3.0.3
```

`3.0.3` 是服务端版本，取决于你起的 Milvus 镜像；只要客户端 `pymilvus` 与服务端同大版本即可。

### 演示语料入库：Embedding → Milvus

用 6 句演示语料建一个 `rag_lab_demo` collection：Embedding 走 Xinference 的 `my_qwen3_embedding_0.6b`（1024 维），索引用 HNSW + COSINE。`drop_old=True` 让课件可以反复重跑而不重复堆数据。

In [8]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_milvus import Milvus

_ = load_dotenv(find_dotenv())

COLLECTION = "rag_lab_demo"

DOCS = [
    "Milvus 是开源向量数据库，支持 HNSW、IVF 等索引与标量过滤。",
    "向量检索一般先召回 top-k 候选，再用 reranker 对候选做精排。",
    "Reranker 常见的接口风格是 Cohere 的 /v1/rerank，Xinference 也兼容。",
    "今天下午下大雨，会议改到线上进行。",
    "Python 的 asyncio 用事件循环并发调度协程任务。",
    "Embedding 的向量维度必须和 Milvus collection 的维度一致。",
]

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

docs = [Document(page_content=text, metadata={"source": f"demo-{i}"}) for i, text in enumerate(DOCS)]

# drop_old=True：课件反复重跑，每次重建演示 collection
vector_store = Milvus.from_documents(
    docs,
    my_emb,
    collection_name=COLLECTION,
    connection_args={"uri": os.environ["MILVUS_URI"]},
    index_params={"metric_type": "COSINE", "index_type": "HNSW", "params": {"M": 16, "efConstruction": 200}},
    search_params={"metric_type": "COSINE", "params": {"ef": 64}},
    drop_old=True,
)

# 行数统计要 flush 之后才准确；client 复用上一段的连接
client.flush(COLLECTION)
print(len(docs), COLLECTION, client.get_collection_stats(COLLECTION))

query = "向量检索召回之后怎么做精排？"

# 粗排（向量召回）：COSINE 相似度，越大越相似
for doc, score in vector_store.similarity_search_with_score(query, k=4):
    print(round(score, 3), doc.page_content)

6 rag_lab_demo {'row_count': 6}
0.845 向量检索一般先召回 top-k 候选，再用 reranker 对候选做精排。
0.611 Reranker 常见的接口风格是 Cohere 的 /v1/rerank，Xinference 也兼容。
0.474 Embedding 的向量维度必须和 Milvus collection 的维度一致。
0.426 Milvus 是开源向量数据库，支持 HNSW、IVF 等索引与标量过滤。


**预期输出：**

```text
6 rag_lab_demo {'row_count': 6}
0.845 向量检索一般先召回 top-k 候选，再用 reranker 对候选做精排。
0.611 Reranker 常见的接口风格是 Cohere 的 /v1/rerank，Xinference 也兼容。
0.474 Embedding 的向量维度必须和 Milvus collection 的维度一致。
0.426 Milvus 是开源向量数据库，支持 HNSW、IVF 等索引与标量过滤。
```

`metric_type="COSINE"` 时 `similarity_search_with_score` 返回的是相似度，越大越相似。召回阶段只看向量距离，最后那句 Milvus 定义也进了候选——它语义相关，但不是这个问题最该进 Prompt 的片段。

### LangChain 里做 Rerank：自定义 DocumentCompressor

先说结论：**LangChain 没有 Xinference 的 rerank 集成**，得自己包一层。实测依据（都在 `env_rag_uv` 里验过）：

- `langchain_community/document_compressors/` 只有 dashscope、flashrank、infinity、jina、openvino、rankllm、volcengine 七个 rerank，没有 xinference。
- `xinference_client` 的 `RESTfulRerankModelHandle` 只有 `rerank()`，不提供 LangChain 封装。
- 最接近的 `langchain_cohere.CohereRerank(base_url=...)` 走不通：cohere SDK 固定请求 `{base_url}/v2/rerank`，Xinference 只暴露 `/v1/rerank`，两种 base_url 写法都返回 `405 Method Not Allowed`。

LangChain 1.x 的两个 import 变化要留意：`ContextualCompressionRetriever` 在 `langchain_classic.retrievers`，`UserDocumentCompressor` 已移除，改成继承 `langchain_core.documents.compressor.BaseDocumentCompressor`。

In [9]:
import os
from typing import Sequence

import httpx
from dotenv import load_dotenv, find_dotenv
from pydantic import Field
from langchain_core.callbacks import Callbacks
from langchain_core.documents import Document
from langchain_core.documents.compressor import BaseDocumentCompressor
from langchain_classic.retrievers import ContextualCompressionRetriever

_ = load_dotenv(find_dotenv())


class XinferenceRerankCompressor(BaseDocumentCompressor):
    """把 Xinference 的 /v1/rerank 包成 LangChain DocumentCompressor。"""

    base_url: str
    api_key: str = ""
    model: str
    top_n: int = 3
    score_field: str = "relevance_score"
    timeout: float = Field(default=60.0)

    def compress_documents(
        self,
        documents: Sequence[Document],
        query: str,
        callbacks: Callbacks | None = None,
    ) -> Sequence[Document]:
        if not documents:
            return []
        resp = httpx.post(
            self.base_url.rstrip("/") + "/rerank",
            headers={"Authorization": "Bearer " + self.api_key},
            json={
                "model": self.model,
                "query": query,
                "documents": [doc.page_content for doc in documents],
                "top_n": min(self.top_n, len(documents)),
            },
            timeout=self.timeout,
        )
        resp.raise_for_status()
        compressed = []
        for item in resp.json()["results"]:
            doc = documents[item["index"]].model_copy(deep=True)
            doc.metadata[self.score_field] = item["relevance_score"]
            compressed.append(doc)
        return compressed


compressor = XinferenceRerankCompressor(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_RERANKER_MODEL_ID"],
    top_n=2,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# 召回 4 条 -> Rerank 取 2 条，最终只有 2 条进 Prompt
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)

for doc in compression_retriever.invoke(query):
    print(round(doc.metadata["relevance_score"], 6), doc.page_content)

1.0 向量检索一般先召回 top-k 候选，再用 reranker 对候选做精排。
0.126953 Reranker 常见的接口风格是 Cohere 的 /v1/rerank，Xinference 也兼容。


**预期输出：**

```text
1.0 向量检索一般先召回 top-k 候选，再用 reranker 对候选做精排。
0.126953 Reranker 常见的接口风格是 Cohere 的 /v1/rerank，Xinference 也兼容。
```

召回 4 条 → Rerank 留 2 条，最终只有这 2 条进 Prompt，`top_n` 就是控制送给大模型片段数量的闸门。精排分数（1.0 / 0.127）比向量相似度（0.845 / 0.611）拉得更开，把"讲机制"的片段和"只是提到关键词"的片段区分开了。